# 🚀 RAG Document Ingestion Pipeline - GCP Version
## From file_to_ingest Folder → GCS → PostgreSQL with pgvector

**Pipeline:**
1. Scan `file_to_ingest/` folder
2. Upload files to GCS (Google Cloud Storage)
3. Extract text from PDF (dengan page tracking)
4. Chunk text (dengan overlap)
5. Generate embeddings (OpenAI 1536-dim)
6. Insert chunks ke PostgreSQL pgvector
7. Test semantic search

## 0️⃣ Setup - Database Connection & Imports

In [1]:
import os
import sys
import subprocess
import json
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple, Optional
import pandas as pd
import uuid
import pdfplumber
import io
import re
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

# Load .env (for Jupyter/local development)
load_dotenv(Path(".env"))

print("✅ All imports loaded successfully\n")

✅ All imports loaded successfully



## 1️⃣ Setup Database Connection & GCS Storage

In [2]:
# ============================================================================
# 🔌 DATABASE CONNECTION (via docker-compose exec + Cloud SQL Proxy)
# ============================================================================

class DatabaseConnection:
    """Database connection via docker-compose exec (same as proven notebook)"""
    def __init__(self, working_dir: str = "."):
        self.working_dir = working_dir
        self.user = "llm_user"
        self.db = "system_llm"
        self.password = "anLLMUser123123"
        self.host = "cloud-sql-proxy"

    def execute_sql(self, query: str, fetch: bool = False) -> str:
        """Execute SQL via docker-compose exec with psql"""
        # Use -f - to read from stdin (psql will read query from stdin)
        cmd = [
            "docker-compose", "-f", "docker-compose.yml",
            "exec", "-T", "api",
            "bash", "-c",
            f'PGPASSWORD="{self.password}" psql -h {self.host} -U {self.user} -d {self.db} -w -t'
            if fetch 
            else f'PGPASSWORD="{self.password}" psql -h {self.host} -U {self.user} -d {self.db} -w'
        ]

        try:
            result = subprocess.run(
                cmd,
                input=query,
                capture_output=True,
                text=True,
                encoding='utf-8',
                errors='replace',
                cwd=self.working_dir
            )

            if result.returncode != 0:
                error_msg = result.stderr.strip() if result.stderr else "Unknown error"
                raise Exception(f"SQL Error: {error_msg}")

            return result.stdout.strip() if fetch else ""

        except Exception as e:
            print(f"Database Error: {e}")
            raise

    def test_connection(self) -> bool:
        """Test database connection"""
        try:
            version = self.execute_sql("SELECT version();", fetch=True)
            if version:
                db_version = version.split(',')[0].strip()
                print(f"✅ Connected to PostgreSQL")
                print(f"   Version: {db_version}\n")
                return True
        except Exception as e:
            print(f"❌ Connection failed: {e}\n")
            return False

# Initialize database connection
db = DatabaseConnection(working_dir=".")
print("=" * 80)
print("DATABASE CONNECTION TEST (via docker-compose + Cloud SQL Proxy)")
print("=" * 80)

if not db.test_connection():
    print("⚠️  Make sure docker-compose is running:")
    print("    docker-compose -f docker-compose.yml up -d\n")

print("All imports and database loaded successfully\n")

DATABASE CONNECTION TEST (via docker-compose + Cloud SQL Proxy)
✅ Connected to PostgreSQL
   Version: PostgreSQL 15.14 on x86_64-pc-linux-gnu

All imports and database loaded successfully



## 2️⃣ Setup GCS Storage

In [3]:
# ============================================================================
# 📦 GCS STORAGE PROVIDER
# ============================================================================

class GCSStorageProvider:
    """Google Cloud Storage Provider for RAG documents"""
    def __init__(self, bucket_name: str, credentials_path: str = None, project_id: str = None):
        from google.cloud import storage as gcs_storage
        from google.oauth2 import service_account
        
        # Remove gs:// prefix if present
        if bucket_name.startswith('gs://'):
            bucket_name = bucket_name[5:]
        
        self.bucket_name = bucket_name
        
        # Initialize GCS client
        if credentials_path and os.path.exists(credentials_path):
            creds = service_account.Credentials.from_service_account_file(credentials_path)
            self.client = gcs_storage.Client(credentials=creds, project=project_id or creds.project_id)
        else:
            # Use default credentials (Application Default Credentials)
            self.client = gcs_storage.Client(project=project_id)
        
        self.bucket = self.client.bucket(self.bucket_name)
    
    def put(self, file_id: str, content: bytes) -> str:
        """Upload file to GCS"""
        blob = self.bucket.blob(f"uploads/{file_id}.pdf")
        blob.upload_from_string(content, content_type="application/pdf")
        return file_id
    
    def get(self, file_id: str) -> bytes:
        """Download file from GCS"""
        blob = self.bucket.blob(f"uploads/{file_id}.pdf")
        if not blob.exists():
            raise FileNotFoundError(f"File not found in GCS: {file_id}")
        return blob.download_as_bytes()

# Initialize GCS Storage
GCS_BUCKET_NAME = os.getenv("GCS_BUCKET_NAME")
GCS_CREDENTIALS_PATH = os.getenv("GCS_CREDENTIALS_PATH")
GCS_PROJECT_ID = os.getenv("GCS_PROJECT_ID")

if not GCS_BUCKET_NAME:
    raise ValueError("GCS_BUCKET_NAME not set in .env")

try:
    storage = GCSStorageProvider(
        GCS_BUCKET_NAME,
        GCS_CREDENTIALS_PATH,
        GCS_PROJECT_ID
    )
    print(f"✅ GCS Storage initialized")
    print(f"   Bucket: {storage.bucket_name}")
    print(f"   Project: {GCS_PROJECT_ID or 'default'}\n")
except Exception as e:
    print(f"❌ GCS initialization failed: {e}")
    print("   Make sure GCS credentials are configured\n")
    raise

✅ GCS Storage initialized
   Bucket: system-llm-storage
   Project: system-llm



## 3️⃣ Discover Files - Scan file_to_ingest Folder

In [4]:
# ============================================================================
# 📁 SCAN file_to_ingest FOLDER
# ============================================================================

ingest_folder = Path("file_to_ingest")

# Create folder if not exists
if not ingest_folder.exists():
    print("📂 Folder file_to_ingest belum ada, membuat folder...")
    ingest_folder.mkdir(parents=True)
else:
    print("📂 Folder file_to_ingest sudah ada")

pdf_files = sorted(ingest_folder.glob("*.pdf"))

print("=" * 80)
print(f"📁 Scanning folder: {ingest_folder.absolute()}")
print("=" * 80 + "\n")

if pdf_files:
    print(f"Found {len(pdf_files)} PDF file(s):\n")
    for i, file_path in enumerate(pdf_files, 1):
        file_size = file_path.stat().st_size
        print(f"  [{i}] {file_path.name}")
        print(f"      Size: {file_size:,} bytes\n")
else:
    print("⚠️  No PDF files found in file_to_ingest folder")
    print("\n📝 Please add PDF files to: file_to_ingest/")
    print("   Then run the next cells to process them.\n")

📂 Folder file_to_ingest sudah ada
📁 Scanning folder: c:\Users\pcgsa\Downloads\system-llm\system-llm-backend\file_to_ingest

Found 55 PDF file(s):

  [1] 489.pdf
      Size: 5,012,574 bytes

  [2] 5.pdf
      Size: 13,018,957 bytes

  [3] 54.pdf
      Size: 12,394,325 bytes

  [4] 6.pdf
      Size: 19,329,521 bytes

  [5] 61.pdf
      Size: 10,809,095 bytes

  [6] [Developer Best Practices] Eric Brechner - Agile Project Management with Kanban (2015, Microsoft Press) - libgen.li.pdf
      Size: 4,179,704 bytes

  [7] [Developer Best Practices] Ken Schwaber - Agile Project Management with Scrum (2004, Microsoft Press) - libgen.li (1).pdf
      Size: 9,629,192 bytes

  [8] [Developer Best Practices] Ken Schwaber - Agile Project Management with Scrum (2004, Microsoft Press) - libgen.li.pdf
      Size: 9,629,192 bytes

  [9] [For Dummies] Mark C. Layton, Steven J Ostermiller, Dean J. Kynaston - Agile Project Management For Dummies (2020, John Wiley & Sons) - libgen.li.pdf
      Size: 10,469,

## 4️⃣ Select Files to Process

In [5]:
# ============================================================================
# 📋 SELECT FILES TO PROCESS
# ============================================================================

# MODIFY THIS: Change which files to process
# Example: [1, 2] to process first and second file
# Example: list(range(1, len(pdf_files) + 1)) to process all
file_indices = list(range(1, len(pdf_files) + 1))  # Process ALL by default

selected_files = []

if pdf_files:
    print("📋 Selected files to ingest:\n")
    for idx in file_indices:
        if 1 <= idx <= len(pdf_files):
            file_path = pdf_files[idx - 1]
            selected_files.append(file_path)
            print(f"  ✅ [{idx}] {file_path.name}")
    
    print(f"\n✅ Total files selected: {len(selected_files)}\n")
else:
    print("❌ No PDF files available to select\n")

📋 Selected files to ingest:

  ✅ [1] 489.pdf
  ✅ [2] 5.pdf
  ✅ [3] 54.pdf
  ✅ [4] 6.pdf
  ✅ [5] 61.pdf
  ✅ [6] [Developer Best Practices] Eric Brechner - Agile Project Management with Kanban (2015, Microsoft Press) - libgen.li.pdf
  ✅ [7] [Developer Best Practices] Ken Schwaber - Agile Project Management with Scrum (2004, Microsoft Press) - libgen.li (1).pdf
  ✅ [8] [Developer Best Practices] Ken Schwaber - Agile Project Management with Scrum (2004, Microsoft Press) - libgen.li.pdf
  ✅ [9] [For Dummies] Mark C. Layton, Steven J Ostermiller, Dean J. Kynaston - Agile Project Management For Dummies (2020, John Wiley & Sons) - libgen.li.pdf
  ✅ [10] [Texts in Computer Science] Peter A. Darnell - C A Software Engineering Approach_ A Software Engineering Approach (2013, Springer) - libgen.li.pdf
  ✅ [11] _ADVAN~1.PDF
  ✅ [12] _GLOBA~1.PDF
  ✅ [13] _OPM3K~1.PDF
  ✅ [14] _SOFTW~1.PDF
  ✅ [15] _TEXTS~2.PDF
  ✅ [16] _UNDER~1.PDF
  ✅ [17] Agile Processes in Software Engineering and Extreme Progra

## 5️⃣ Upload Files & Create DB Records

In [6]:
# ============================================================================
# 📤 UPLOAD FILES TO GCS & CREATE DB RECORDS
# ============================================================================

uploaded_documents = []

if selected_files:
    print("=" * 80)
    print(f"Uploading {len(selected_files)} file(s) to GCS")
    print("=" * 80 + "\n")
    
    # Get a valid user_id from database (or use default)
    try:
        user_result = db.execute_sql("SELECT id FROM \"user\" LIMIT 1;", fetch=True)
        if user_result and user_result.strip():
            user_id = user_result.strip()
        else:
            user_id = "00000000-0000-0000-0000-000000000000"
    except:
        user_id = "00000000-0000-0000-0000-000000000000"
    
    for file_idx, file_path in enumerate(selected_files, 1):
        try:
            # Read file
            file_content = file_path.read_bytes()
            file_size = len(file_content)
            
            # Generate storage filename
            storage_filename = str(uuid.uuid4())
            gcs_path = f"gs://{storage.bucket_name}/uploads/{storage_filename}.pdf"
            
            # Upload to GCS
            storage.put(storage_filename, file_content)
            
            # Create database record
            db_id = str(uuid.uuid4())
            
            insert_query = f"""
            INSERT INTO document (id, user_id, original_filename, filename, file_path, file_size, status, mime_type)
            VALUES ('{db_id}', '{user_id}', '{file_path.name}', '{storage_filename}.pdf', '{gcs_path}', {file_size}, 'UPLOADED', 'application/pdf');
            """
            
            db.execute_sql(insert_query)
            
            uploaded_documents.append({
                "index": file_idx,
                "db_id": db_id,
                "storage_filename": storage_filename,
                "original_filename": file_path.name,
                "file_size": file_size
            })
            
            print(f"  ✅ [{file_idx}] {file_path.name}")
            print(f"      Size: {file_size:,} bytes")
            print(f"      GCS ID: {storage_filename}")
            print(f"      DB ID: {db_id[:8]}...\n")
        
        except Exception as e:
            print(f"  ❌ [{file_idx}] {file_path.name}: {e}\n")
    
    if uploaded_documents:
        print("=" * 80)
        print(f"SUCCESS: {len(uploaded_documents)}/{len(selected_files)} file(s) uploaded")
        print("=" * 80 + "\n")
    else:
        print("FAILED: No files uploaded\n")
else:
    print("No files selected for upload\n")

Uploading 55 file(s) to GCS

  ✅ [1] 489.pdf
      Size: 5,012,574 bytes
      GCS ID: 5b9c7907-fd66-444c-965d-441143c575b0
      DB ID: 8b24f2e0...

  ✅ [2] 5.pdf
      Size: 13,018,957 bytes
      GCS ID: 311cde9d-7b3f-4a8e-ad3b-cda38f7e924d
      DB ID: fff0c25d...

  ✅ [3] 54.pdf
      Size: 12,394,325 bytes
      GCS ID: 875c1d6e-e094-4b86-9baf-c4c086e7cdfe
      DB ID: 825d702b...

  ✅ [4] 6.pdf
      Size: 19,329,521 bytes
      GCS ID: f1093cf6-557e-49fc-9fa0-275ebe5963d7
      DB ID: 0de4c771...

  ✅ [5] 61.pdf
      Size: 10,809,095 bytes
      GCS ID: cde31d45-0260-4f09-9f75-271b67ade1a2
      DB ID: 1d9766f2...

  ✅ [6] [Developer Best Practices] Eric Brechner - Agile Project Management with Kanban (2015, Microsoft Press) - libgen.li.pdf
      Size: 4,179,704 bytes
      GCS ID: 254f93b2-dfe3-4e77-b191-51f3c1263dba
      DB ID: 88366e1c...

  ✅ [7] [Developer Best Practices] Ken Schwaber - Agile Project Management with Scrum (2004, Microsoft Press) - libgen.li (1).pdf
     

In [7]:
# # Clear all files in GCS uploads folder
# from google.cloud import storage as gcs_storage

# bucket_name = "system-llm-storage"
# client = gcs_storage.Client(project="system-llm")
# bucket = client.bucket(bucket_name)

# # Delete all files in uploads/ folder
# blobs = bucket.list_blobs(prefix="uploads/")
# for blob in blobs:
#     blob.delete()
#     print(f"Deleted: {blob.name}")

# print(f"✅ GCS storage cleared - {bucket_name}/uploads/")
# # Execute ini via notebook atau command line:

# # Option A: Via notebook (using db connection)
# db.execute_sql("DELETE FROM document_chunk;")
# db.execute_sql("DELETE FROM document;")
# print("✅ Database cleared - document & document_chunk")

# # Option B: Via bash (direct psql)


## 6️⃣ Extract Text from PDF

In [8]:
# ============================================================================
# 📄 EXTRACT TEXT FROM PDF (mencari berdasarkan original_filename)
# ============================================================================

import fitz  # pip install pymupdf

def extract_text_pymupdf(pdf_bytes: bytes) -> dict:
    """Extract text from PDF with page tracking"""
    pages_text = {}
    try:
        doc = fitz.open(stream=pdf_bytes, filetype="pdf")
        for i, page in enumerate(doc, 1):
            text = page.get_text("text")
            if text and text.strip():
                pages_text[i] = text
    except Exception as e:
        print(f"Error extracting PDF: {e}")
        raise
    return pages_text

def get_file_from_gcs_by_filename(original_filename: str):
    """
    Get file dari GCS berdasarkan original_filename.
    Query database untuk mencari storage_filename/file_path.
    """
    try:
        # Query database untuk cari file berdasarkan original_filename
        safe_filename = original_filename.replace("'", "''")
        query = f"SELECT filename FROM document WHERE original_filename = '{safe_filename}' LIMIT 1;"
        
        storage_filename = db.execute_sql(query, fetch=True)
        
        if storage_filename and storage_filename.strip():
            storage_filename = storage_filename.strip()
            if storage_filename.endswith('.pdf'):
                storage_filename = storage_filename[:-4]
            
            # Download dari GCS
            pdf_bytes = storage.get(storage_filename)
            return pdf_bytes
        else:
            return None
            
    except Exception as e:
        print(f"    Error finding file: {e}")
        return None

print("✅ PDF extraction function loaded (search by original_filename)\n")

# Extract from uploaded documents - HYBRID APPROACH
extracted_texts = {}

# Approach 1: Jika uploaded_documents ada (same notebook run)
if uploaded_documents:
    print("=" * 80)
    print(f"Extracting text from {len(uploaded_documents)} document(s) - Same run")
    print("=" * 80 + "\n")
    
    for doc in uploaded_documents:
        db_id = doc["db_id"]
        original_filename = doc["original_filename"]
        storage_filename = doc["storage_filename"]
        
        try:
            pdf_bytes = storage.get(storage_filename)
            pages_text = extract_text_pymupdf(pdf_bytes)
            extracted_texts[db_id] = pages_text
            
            total_chars = sum(len(t) for t in pages_text.values())
            print(f"  ✅ {original_filename}: {len(pages_text)} pages, {total_chars} characters")
        except Exception as e:
            print(f"  ❌ {original_filename}: {e}")
    
    if extracted_texts:
        print(f"\n✅ Extracted {len(extracted_texts)} documents\n")
    else:
        print(f"❌ No documents extracted\n")

# Approach 2: Jika re-run (no uploaded_documents), query database
else:
    print("=" * 80)
    print("Re-run mode: Query database for existing files...")
    print("=" * 80 + "\n")
    
    # Query all documents from database
    query = "SELECT id, original_filename FROM document ORDER BY uploaded_at DESC LIMIT 100;"
    result = db.execute_sql(query, fetch=True)
    
    print(f"DEBUG: Query result type: {type(result)}, length: {len(result) if result else 0}")
    print(f"DEBUG: First 200 chars: {result[:200] if result else 'None'}\n")
    
    if result and result.strip():
        documents = []
        
        # Parse hasil query - skip header jika ada
        lines = result.split('\n')
        print(f"DEBUG: Total lines: {len(lines)}\n")
        
        for idx, line in enumerate(lines):
            # Skip empty lines dan header lines (dengan dashes)
            if not line.strip() or '---' in line or '(' in line:
                continue
            
            if '|' in line:
                parts = [p.strip() for p in line.split('|')]
                if len(parts) >= 2:
                    doc_id = parts[0]
                    filename = parts[1]
                    
                    # Validate doc_id looks like UUID
                    if doc_id and len(doc_id) > 10 and filename:
                        documents.append({"db_id": doc_id, "original_filename": filename})
                        print(f"DEBUG: Parsed - ID: {doc_id[:8]}..., File: {filename}")
        
        print(f"\nFound {len(documents)} document(s)\n")
        
        if documents:
            for doc in documents:
                db_id = doc["db_id"]
                original_filename = doc["original_filename"]
                
                try:
                    print(f"  Extracting: {original_filename}")
                    pdf_bytes = get_file_from_gcs_by_filename(original_filename)
                    
                    if pdf_bytes:
                        pages_text = extract_text_pymupdf(pdf_bytes)
                        extracted_texts[db_id] = pages_text
                        
                        total_chars = sum(len(t) for t in pages_text.values())
                        print(f"    ✅ {len(pages_text)} pages, {total_chars} characters\n")
                    else:
                        print(f"    ❌ Could not retrieve from GCS\n")
                        
                except Exception as e:
                    print(f"    ❌ {e}\n")
            
            if extracted_texts:
                print(f"✅ Extracted {len(extracted_texts)} documents\n")
            else:
                print(f"❌ No documents extracted successfully\n")
    else:
        print("❌ No documents found in database or empty result\n")

✅ PDF extraction function loaded (search by original_filename)

Extracting text from 55 document(s) - Same run

  ✅ 489.pdf: 298 pages, 608493 characters
  ✅ 5.pdf: 253 pages, 481860 characters
  ✅ 54.pdf: 352 pages, 735334 characters
  ✅ 6.pdf: 647 pages, 1720253 characters
  ✅ 61.pdf: 338 pages, 357827 characters
  ✅ [Developer Best Practices] Eric Brechner - Agile Project Management with Kanban (2015, Microsoft Press) - libgen.li.pdf: 167 pages, 347902 characters
  ✅ [Developer Best Practices] Ken Schwaber - Agile Project Management with Scrum (2004, Microsoft Press) - libgen.li (1).pdf: 178 pages, 432494 characters
  ✅ [Developer Best Practices] Ken Schwaber - Agile Project Management with Scrum (2004, Microsoft Press) - libgen.li.pdf: 178 pages, 432494 characters
  ✅ [For Dummies] Mark C. Layton, Steven J Ostermiller, Dean J. Kynaston - Agile Project Management For Dummies (2020, John Wiley & Sons) - libgen.li.pdf: 478 pages, 969219 characters
  ✅ [Texts in Computer Science] Peter

## 7️⃣ Execute Text Chunking

In [9]:
# ============================================================================
# 📦 TEXT CHUNKING (Global chunking with page tracking)
# ============================================================================

def chunk_text_with_pages(
    pages_text: Dict[int, str],
    chunk_size: int = 500,
    overlap: int = 50
) -> List[Tuple[str, int, int]]:
    """
    Chunk text globally (not per-page) while tracking page numbers.
    Returns: List of (chunk_content, start_page, end_page)

    Algorithm:
    1. Combine all pages into one text stream
    2. Split by sentences
    3. Build chunks from sentences (each chunk can span multiple pages)
    4. Track which pages each chunk touches
    """
    chunks_with_pages = []

    # Convert to list of (sentence, page_number) tuples
    all_sentences = []
    for page_num in sorted(pages_text.keys()):
        page_content = pages_text[page_num]
        sentences = re.split(r'(?<=[.!?])\s+', page_content)
        for sentence in sentences:
            if sentence.strip():
                all_sentences.append((sentence, page_num))

    if not all_sentences:
        return chunks_with_pages

    # Build chunks globally
    current_chunk = []
    current_pages = set()
    current_size = 0

    for sentence, page_num in all_sentences:
        words = sentence.split()
        if not words:
            continue

        # If adding this sentence exceeds chunk_size AND we have content, save chunk
        if current_size + len(words) > chunk_size and current_chunk:
            chunk_content = ' '.join(current_chunk)
            start_page = min(current_pages)
            end_page = max(current_pages)
            chunks_with_pages.append((chunk_content, start_page, end_page))

            # OVERLAP: Keep last N words
            overlap_words = current_chunk[-overlap:] if len(current_chunk) > overlap else current_chunk
            current_chunk = overlap_words
            current_size = len(' '.join(current_chunk).split())
            current_pages = {page_num}

        current_chunk.extend(words)
        current_pages.add(page_num)
        current_size += len(words)

    # Save remaining chunk
    if current_chunk:
        chunk_content = ' '.join(current_chunk)
        start_page = min(current_pages)
        end_page = max(current_pages)
        chunks_with_pages.append((chunk_content, start_page, end_page))

    return chunks_with_pages

print("✅ Text chunking function loaded (GLOBAL CHUNKING - spans multiple pages)\n")

# Create chunks
chunks_by_document = {}  # {db_id: [(content, start_page, end_page), ...]}

if extracted_texts:
    print("=" * 80)
    print(f"Creating chunks for {len(extracted_texts)} document(s)")
    print("=" * 80 + "\n")
    
    for doc_id, pages_text in extracted_texts.items():
        chunks_with_pages = chunk_text_with_pages(pages_text, chunk_size=500, overlap=50)
        chunks_by_document[doc_id] = chunks_with_pages
        
        doc = next((d for d in uploaded_documents if d["db_id"] == doc_id), None)
        if doc:
            print(f"  📄 {doc['original_filename']}")
            print(f"     Total Pages: {len(pages_text)}")
            print(f"     Total Chunks: {len(chunks_with_pages)}")
            
            # Analyze chunk distribution
            single_page = sum(1 for _, start, end in chunks_with_pages if start == end)
            multi_page = sum(1 for _, start, end in chunks_with_pages if start != end)
            
            print(f"     Single-page chunks: {single_page}")
            print(f"     Multi-page chunks: {multi_page}")
            
            if chunks_with_pages:
                avg_words = sum(len(c[0].split()) for c in chunks_with_pages) / len(chunks_with_pages)
                print(f"     Avg words/chunk: {avg_words:.0f}\n")
    
    total = sum(len(c) for c in chunks_by_document.values())
    print("=" * 80)
    print(f"SUCCESS: Created {total} chunks")
    print("=" * 80 + "\n")
else:
    print("No text to chunk\n")

✅ Text chunking function loaded (GLOBAL CHUNKING - spans multiple pages)

Creating chunks for 55 document(s)

  📄 489.pdf
     Total Pages: 298
     Total Chunks: 214
     Single-page chunks: 3
     Multi-page chunks: 211
     Avg words/chunk: 482

  📄 5.pdf
     Total Pages: 253
     Total Chunks: 180
     Single-page chunks: 16
     Multi-page chunks: 164
     Avg words/chunk: 486

  📄 54.pdf
     Total Pages: 352
     Total Chunks: 284
     Single-page chunks: 26
     Multi-page chunks: 258
     Avg words/chunk: 486

  📄 6.pdf
     Total Pages: 647
     Total Chunks: 613
     Single-page chunks: 74
     Multi-page chunks: 539
     Avg words/chunk: 486

  📄 61.pdf
     Total Pages: 338
     Total Chunks: 127
     Single-page chunks: 0
     Multi-page chunks: 127
     Avg words/chunk: 484

  📄 [Developer Best Practices] Eric Brechner - Agile Project Management with Kanban (2015, Microsoft Press) - libgen.li.pdf
     Total Pages: 167
     Total Chunks: 129
     Single-page chunks: 9
  

## 8️⃣ Load OpenAI & Generate Embeddings

In [10]:
# ============================================================================
# 🔗 OPENAI EMBEDDINGS
# ============================================================================

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    print("ERROR: OPENAI_API_KEY not set in .env")
    print("Make sure .env has: OPENAI_API_KEY=sk-proj-...")
    raise ValueError("OPENAI_API_KEY not set")

client = OpenAI(api_key=OPENAI_API_KEY)

def generate_embedding(text: str) -> List[float]:
    """Generate 1536-dimensional embedding using OpenAI"""
    response = client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

print("✅ OpenAI embedding function loaded (1536 dimensions)")
print(f"Using API key: {OPENAI_API_KEY[:20]}...\n")

if chunks_by_document:
    print("Testing embedding generation...")
    
    first_doc_id = list(chunks_by_document.keys())[0]
    first_chunk = chunks_by_document[first_doc_id][0][0]
    
    try:
        embedding = generate_embedding(first_chunk[:1000])
        print(f"  OK: Generated {len(embedding)}-dimensional embedding")
        print(f"      First 5 values: {embedding[:5]}\n")
    except Exception as e:
        print(f"  ERROR: {e}\n")
        raise
else:
    print("No chunks for embedding test\n")

✅ OpenAI embedding function loaded (1536 dimensions)
Using API key: sk-proj-pH9p6un0rQCs...

Testing embedding generation...
  OK: Generated 1536-dimensional embedding
      First 5 values: [0.013762412592768669, 0.01353723369538784, -0.0416448749601841, 0.0007674299995414913, 0.012172913178801537]



## 9️⃣ Embedding Cost Estimation

In [11]:
# ============================================================================
# 💰 EMBEDDING COST ESTIMATION
# ============================================================================

PRICE_PER_1M_TOKENS = 0.02  # USD per 1M tokens for text-embedding-3-small

print("=" * 80)
print("💰 EMBEDDING COST CONFIGURATION")
print("=" * 80)
print(f"Model: text-embedding-3-small")
print(f"Price: ${PRICE_PER_1M_TOKENS} per 1M input tokens")
print(f"\n✏️  To change price, modify PRICE_PER_1M_TOKENS variable above\n")

def estimate_tokens(text: str) -> int:
    """Estimate token count for text (roughly 1 token per 4 characters)"""
    return len(text) // 4

def estimate_embedding_cost(chunks_data: dict) -> dict:
    """Estimate total cost for embedding all chunks"""
    total_tokens = 0
    total_chunks = 0
    breakdown = []
    
    for doc_id, chunks_with_pages in chunks_data.items():
        doc = next((d for d in uploaded_documents if d["db_id"] == doc_id), None)
        filename = doc["original_filename"] if doc else "Unknown"
        
        doc_tokens = 0
        doc_chunk_count = len(chunks_with_pages)
        
        for chunk_content, _, _ in chunks_with_pages:
            tokens = estimate_tokens(chunk_content)
            doc_tokens += tokens
        
        doc_cost = (doc_tokens / 1_000_000) * PRICE_PER_1M_TOKENS
        breakdown.append((filename, doc_chunk_count, doc_tokens, doc_cost))
        
        total_tokens += doc_tokens
        total_chunks += doc_chunk_count
    
    total_cost = (total_tokens / 1_000_000) * PRICE_PER_1M_TOKENS
    
    return {
        "total_chunks": total_chunks,
        "estimated_tokens": total_tokens,
        "estimated_cost_usd": total_cost,
        "chunks_breakdown": breakdown
    }

print("Cost estimation function loaded\n")

💰 EMBEDDING COST CONFIGURATION
Model: text-embedding-3-small
Price: $0.02 per 1M input tokens

✏️  To change price, modify PRICE_PER_1M_TOKENS variable above

Cost estimation function loaded



## 🔟 Display Cost Estimate

In [12]:
# ============================================================================
# 📊 DISPLAY COST ESTIMATE BEFORE EMBEDDING
# ============================================================================

if chunks_by_document:
    print("=" * 80)
    print("📊 EMBEDDING COST ESTIMATION")
    print("=" * 80 + "\n")
    
    # Calculate estimate
    cost_estimate = estimate_embedding_cost(chunks_by_document)
    
    # Show breakdown per document
    print("📄 Cost Breakdown by Document:\n")
    for filename, chunk_count, tokens, cost in cost_estimate["chunks_breakdown"]:
        print(f"  {filename}")
        print(f"    Chunks: {chunk_count}")
        print(f"    Est. Tokens: {tokens:,}")
        print(f"    Est. Cost: ${cost:.6f}\n")
    
    # Show summary
    print("=" * 80)
    print("📋 TOTAL ESTIMATE")
    print("=" * 80)
    print(f"Total Chunks: {cost_estimate['total_chunks']}")
    print(f"Total Est. Tokens: {cost_estimate['estimated_tokens']:,}")
    print(f"Price per 1M Tokens: ${PRICE_PER_1M_TOKENS}")
    print(f"\n💰 TOTAL ESTIMATED COST: ${cost_estimate['estimated_cost_usd']:.6f}")
    print("=" * 80)
    print("\n⚠️  This is an estimate. Actual cost may vary based on OpenAI's tokenization.")
    print("    Run the next cell to proceed with embedding generation.\n")
else:
    print("⚠️  No chunks available for cost estimation\n")

📊 EMBEDDING COST ESTIMATION

📄 Cost Breakdown by Document:

  489.pdf
    Chunks: 214
    Est. Tokens: 169,388
    Est. Cost: $0.003388

  5.pdf
    Chunks: 180
    Est. Tokens: 134,210
    Est. Cost: $0.002684

  54.pdf
    Chunks: 284
    Est. Tokens: 199,106
    Est. Cost: $0.003982

  6.pdf
    Chunks: 613
    Est. Tokens: 472,591
    Est. Cost: $0.009452

  61.pdf
    Chunks: 127
    Est. Tokens: 97,346
    Est. Cost: $0.001947

  [Developer Best Practices] Eric Brechner - Agile Project Management with Kanban (2015, Microsoft Press) - libgen.li.pdf
    Chunks: 129
    Est. Tokens: 96,772
    Est. Cost: $0.001935

  [Developer Best Practices] Ken Schwaber - Agile Project Management with Scrum (2004, Microsoft Press) - libgen.li (1).pdf
    Chunks: 156
    Est. Tokens: 119,830
    Est. Cost: $0.002397

  [Developer Best Practices] Ken Schwaber - Agile Project Management with Scrum (2004, Microsoft Press) - libgen.li.pdf
    Chunks: 156
    Est. Tokens: 119,830
    Est. Cost: $0.0023

## 1️⃣1️⃣ Insert Chunks to PostgreSQL with Embeddings

In [13]:
# ============================================================================
# 📤 INSERT CHUNKS TO POSTGRESQL
# ============================================================================

def insert_chunks_to_db(document_id: str, chunks_with_pages: List[Tuple[str, int, int]]):
    """
    Insert chunks with embeddings to PostgreSQL pgvector.
    Supports chunks spanning multiple pages.
    """
    try:
        # Update status to PROCESSING
        status_query = f"UPDATE document SET status = 'PROCESSING' WHERE id = '{document_id}';"
        db.execute_sql(status_query)
        print(f"  Status: PROCESSING")
        
        # Insert chunks
        print(f"  Inserting {len(chunks_with_pages)} chunks...")
        
        for idx, (chunk_content, start_page, end_page) in enumerate(chunks_with_pages):
            # Generate embedding
            embedding = generate_embedding(chunk_content)
            embedding_json = json.dumps(embedding)
            
            # Escape quotes for SQL
            safe_content = chunk_content.replace("'", "''")
            safe_embedding = embedding_json.replace("'", "''")
            
            # Store page range in metadata
            metadata = json.dumps({"start_page": start_page, "end_page": end_page})
            safe_metadata = metadata.replace("'", "''")
            
            # Insert query
            chunk_id = str(uuid.uuid4())
            insert_query = f"""
            INSERT INTO document_chunk
            (id, document_id, chunk_index, content, page_number, embedding, chunk_metadata, created_at)
            VALUES
            ('{chunk_id}', '{document_id}', {idx}, '{safe_content}', {start_page}, '{safe_embedding}', '{safe_metadata}'::jsonb, now());
            """
            
            db.execute_sql(insert_query)
            
            if (idx + 1) % 10 == 0:
                print(f"     Progress: {idx + 1}/{len(chunks_with_pages)}")
        
        # Update status to PROCESSED
        processed_query = f"UPDATE document SET status = 'PROCESSED', processed_at = now() WHERE id = '{document_id}';"
        db.execute_sql(processed_query)
        print(f"  Status: PROCESSED")
        
    except Exception as e:
        print(f"  ERROR: {e}")
        raise

print("Insert function loaded (handles multi-page chunks)\n")

# Process all documents
if uploaded_documents and chunks_by_document:
    print("=" * 80)
    print(f"Starting ingestion for {len(uploaded_documents)} document(s)")
    print("=" * 80 + "\n")
    
    for doc_idx, doc in enumerate(uploaded_documents, 1):
        document_id = doc["db_id"]
        filename = doc["original_filename"]
        
        if document_id not in chunks_by_document:
            print(f"Skipping {filename} - no chunks\n")
            continue
        
        try:
            chunks_with_pages = chunks_by_document[document_id]
            print(f"[{doc_idx}/{len(uploaded_documents)}] {filename}")
            insert_chunks_to_db(document_id, chunks_with_pages)
            print()
        except Exception as e:
            print(f"FAILED: {e}\n")
    
    print("=" * 80)
    print("INGESTION COMPLETE!")
    print("=" * 80 + "\n")
else:
    print("Missing prerequisites\n")

Insert function loaded (handles multi-page chunks)

Starting ingestion for 55 document(s)

[1/55] 489.pdf
  Status: PROCESSING
  Inserting 214 chunks...
     Progress: 10/214
     Progress: 20/214
     Progress: 30/214
     Progress: 40/214
     Progress: 50/214
     Progress: 60/214
     Progress: 70/214
     Progress: 80/214
     Progress: 90/214
     Progress: 100/214
     Progress: 110/214
     Progress: 120/214
     Progress: 130/214
     Progress: 140/214
     Progress: 150/214
     Progress: 160/214
     Progress: 170/214
     Progress: 180/214
     Progress: 190/214
     Progress: 200/214
     Progress: 210/214
  Status: PROCESSED

[2/55] 5.pdf
  Status: PROCESSING
  Inserting 180 chunks...
     Progress: 10/180
     Progress: 20/180
     Progress: 30/180
     Progress: 40/180
     Progress: 50/180
     Progress: 60/180
     Progress: 70/180
     Progress: 80/180
     Progress: 90/180
     Progress: 100/180
     Progress: 110/180
     Progress: 120/180
     Progress: 130/180
  

In [ ]:
# Identifikasi dokumen yang belum diproses
failed_docs = []
for doc in uploaded_documents:
    doc_id = doc["db_id"]
    if doc_id in chunks_by_document:
        chunks_with_pages = chunks_by_document[doc_id]
        
        # Cek apakah sudah ada chunks di database
        count_query = f"SELECT COUNT(*) FROM document_chunk WHERE document_id = '{doc_id}';"
        count = db.execute_sql(count_query, fetch=True).strip()
        
        if not count or count == "0":
            failed_docs.append({
                "db_id": doc_id,
                "original_filename": doc["original_filename"],
                "chunks": chunks_with_pages
            })

print(f"Found {len(failed_docs)} documents that need processing\n")

# Process only failed documents
if failed_docs:
    print("=" * 80)
    print(f"Retrying {len(failed_docs)} failed document(s)")
    print("=" * 80 + "\n")
    
    for doc_idx, doc in enumerate(failed_docs, 1):
        document_id = doc["db_id"]
        filename = doc["original_filename"]
        chunks_with_pages = doc["chunks"]
        
        try:
            print(f"[{doc_idx}/{len(failed_docs)}] {filename}")
            insert_chunks_to_db(document_id, chunks_with_pages)
            print()
        except Exception as e:
            print(f"FAILED: {e}\n")
    
    print("=" * 80)
    print("RETRY COMPLETE!")
    print("=" * 80 + "\n")
else:
    print("All documents already processed\n")

Found 32 documents that need processing

Retrying 32 failed document(s)

[1/32] ELVIS C.. TOWLE JR. BRADFORD A. FOSTER - SOFTWARE ENGINEERING _ a methodical approach, (2021, CRC PRESS) - libgen.li.pdf
  Status: PROCESSING
  Inserting 303 chunks...
     Progress: 10/303
     Progress: 20/303
     Progress: 30/303
     Progress: 40/303
     Progress: 50/303
     Progress: 60/303
     Progress: 70/303
     Progress: 80/303
     Progress: 90/303
     Progress: 100/303
     Progress: 110/303
     Progress: 120/303
     Progress: 130/303
     Progress: 140/303
     Progress: 150/303
     Progress: 160/303
     Progress: 170/303
     Progress: 180/303
     Progress: 190/303
     Progress: 200/303
     Progress: 210/303
     Progress: 220/303
     Progress: 230/303
     Progress: 240/303
     Progress: 250/303
     Progress: 260/303
     Progress: 270/303
     Progress: 280/303
     Progress: 290/303
     Progress: 300/303
  Status: PROCESSED

[2/32] Eric J. Braude, Michael E. Bernstein - Soft

In [14]:
# ============================================================================
# 🧹 CLEAN TEXT FOR DATABASE (Handle Unicode ligatures & special chars)
# ============================================================================

def clean_text_for_db(text: str) -> str:
    """
    Clean problematic Unicode characters before inserting to DB.
    Handles PDF ligatures and other special characters that cause encoding issues.
    """
    # Replace common PDF ligatures
    replacements = {
        '\ufb00': 'ff',  # ﬀ -> ff
        '\ufb01': 'fi',  # ﬁ -> fi
        '\ufb02': 'fl',  # ﬂ -> fl
        '\ufb03': 'ffi', # ﬃ -> ffi
        '\ufb04': 'ffl', # ﬄ -> ffl
    }
    
    for old, new in replacements.items():
        text = text.replace(old, new)
    
    return text

print("✅ Text cleaning function loaded (handles Unicode ligatures)\n")

✅ Text cleaning function loaded (handles Unicode ligatures)



In [15]:
# ============================================================================
# 📤 INSERT CHUNKS TO POSTGRESQL
# ============================================================================

def insert_chunks_to_db(document_id: str, chunks_with_pages: List[Tuple[str, int, int]]):
    """
    Insert chunks with embeddings to PostgreSQL pgvector.
    Supports chunks spanning multiple pages.
    """
    try:
        # Update status to PROCESSING
        status_query = f"UPDATE document SET status = 'PROCESSING' WHERE id = '{document_id}';"
        db.execute_sql(status_query)
        print(f"  Status: PROCESSING")
        
        # Insert chunks
        print(f"  Inserting {len(chunks_with_pages)} chunks...")
        
        for idx, (chunk_content, start_page, end_page) in enumerate(chunks_with_pages):
            # Clean text for database (handle Unicode ligatures)
            chunk_content = clean_text_for_db(chunk_content)
            
            # Generate embedding
            embedding = generate_embedding(chunk_content)
            embedding_json = json.dumps(embedding)
            
            # Escape quotes for SQL
            safe_content = chunk_content.replace("'", "''")
            safe_embedding = embedding_json.replace("'", "''")
            
            # Store page range in metadata
            metadata = json.dumps({"start_page": start_page, "end_page": end_page})
            safe_metadata = metadata.replace("'", "''")
            
            # Insert query
            chunk_id = str(uuid.uuid4())
            insert_query = f"""
            INSERT INTO document_chunk
            (id, document_id, chunk_index, content, page_number, embedding, chunk_metadata, created_at)
            VALUES
            ('{chunk_id}', '{document_id}', {idx}, '{safe_content}', {start_page}, '{safe_embedding}', '{safe_metadata}'::jsonb, now());
            """
            
            db.execute_sql(insert_query)
            
            if (idx + 1) % 10 == 0:
                print(f"     Progress: {idx + 1}/{len(chunks_with_pages)}")
        
        # Update status to PROCESSED
        processed_query = f"UPDATE document SET status = 'PROCESSED', processed_at = now() WHERE id = '{document_id}';"
        db.execute_sql(processed_query)
        print(f"  Status: PROCESSED")
        
    except Exception as e:
        print(f"  ERROR: {e}")
        raise

print("Insert function loaded (handles multi-page chunks + Unicode)\n")

# Process all documents
if uploaded_documents and chunks_by_document:
    print("=" * 80)
    print(f"Starting ingestion for {len(uploaded_documents)} document(s)")
    print("=" * 80 + "\n")
    
    for doc_idx, doc in enumerate(uploaded_documents, 1):
        document_id = doc["db_id"]
        filename = doc["original_filename"]
        
        if document_id not in chunks_by_document:
            print(f"Skipping {filename} - no chunks\n")
            continue
        
        try:
            chunks_with_pages = chunks_by_document[document_id]
            print(f"[{doc_idx}/{len(uploaded_documents)}] {filename}")
            insert_chunks_to_db(document_id, chunks_with_pages)
            print()
        except Exception as e:
            print(f"FAILED: {e}\n")
    
    print("=" * 80)
    print("INGESTION COMPLETE!")
    print("=" * 80 + "\n")
else:
    print("Missing prerequisites\n")

Insert function loaded (handles multi-page chunks + Unicode)

Starting ingestion for 55 document(s)

[1/55] 489.pdf
Database Error: SQL Error: time="2026-01-08T09:05:36+07:00" level=warning msg="c:\\Users\\pcgsa\\Downloads\\system-llm\\system-llm-backend\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
psql: error: connection to server at "cloud-sql-proxy" (172.21.0.2), port 5432 failed: server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.
  ERROR: SQL Error: time="2026-01-08T09:05:36+07:00" level=warning msg="c:\\Users\\pcgsa\\Downloads\\system-llm\\system-llm-backend\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
psql: error: connection to server at "cloud-sql-proxy" (172.21.0.2), port 5432 failed: server closed the connection unexpectedly
	This prob

## 1️⃣3️⃣ Semantic Search Test

In [16]:
# ============================================================================
# 🔍 SEMANTIC SEARCH TEST
# ============================================================================

def semantic_search(query_text: str, top_k: int = 5) -> list:
    """Semantic search using cosine similarity"""
    try:
        # Generate query embedding
        query_embedding = np.array(generate_embedding(query_text))
        
        # Get chunks from database
        search_query = """
        SELECT dc.content, d.original_filename, dc.page_number, dc.embedding
        FROM document_chunk dc
        JOIN document d ON dc.document_id = d.id
        LIMIT 100;
        """
        result_text = db.execute_sql(search_query, fetch=True)
        
        similarities = []
        if result_text:
            for line in result_text.split('\n'):
                if '|' in line:
                    parts = line.split('|')
                    if len(parts) >= 4:
                        try:
                            content = parts[0].strip()
                            filename = parts[1].strip()
                            page_num = int(parts[2].strip()) if parts[2].strip().isdigit() else 0
                            embedding_json = parts[3].strip()
                            
                            chunk_embedding = np.array(json.loads(embedding_json))
                            similarity = np.dot(query_embedding, chunk_embedding) / (
                                np.linalg.norm(query_embedding) * np.linalg.norm(chunk_embedding) + 1e-10
                            )
                            similarities.append((content[:300], filename, page_num, float(similarity)))
                        except:
                            pass
        
        similarities.sort(key=lambda x: x[3], reverse=True)
        return similarities[:top_k]
    
    except Exception as e:
        print(f"Error: {e}")
        return []

# Test search
print("=" * 80)
print("SEMANTIC SEARCH TEST")
print("=" * 80 + "\n")

QUERY = "Software Engineering"

print(f"Query: '{QUERY}'\n")

results = semantic_search(QUERY, top_k=5)

if results:
    print(f"Found {len(results)} results:\n")
    for i, (content, filename, page, similarity) in enumerate(results, 1):
        print(f"  [{i}] {filename} (page {page})")
        print(f"      Similarity: {similarity:.4f}")
        print(f"      Content: {content[:150]}...\n")
else:
    print("No results found\n")

print("=" * 80)

SEMANTIC SEARCH TEST

Query: 'Software Engineering'

Error: Connection error.
No results found

